# M5 · Inside the machine 4: from vectors to the next character

**Tiny Transformer, episode M5.** After the block, each row is still 28 numbers. Here you run the model's last stages on the
line *“You taught me language; and my p”*: the final LayerNorm, the logits (each row scored against all 65 characters, with the same token table the pass
starts with) and softmax. Then you see that every position predicts, and you run the whole forward pass yourself, stage by
stage with every shape printed, until it equals the model's own output.

[Open in Google Colab](https://colab.research.google.com/github/gkiri/tiny_repo/blob/main/tiny-transformer/notebooks/M5-inside-readout.ipynb) (execution uses Google's hosted runtime; no graphics card needed). On your own computer:
`pip install -r requirements.txt`, then `jupyter notebook M5-inside-readout.ipynb`.

Every number here is computed in this notebook by the trained model: the course's model code with the recorded run's
9,417 learned numbers (`prompter.json`, step 98,000). The numbers the course's page shows are
**Recorded** in the course's export of this line; each cell checks that its own result matches the record to within 0.0002.

In [1]:
#@title Setup: run me first (one click) { display-mode: "form" }
# In Google Colab this fetches the course's labs (notebooks, data and the trained model) and moves into this notebook's
# folder, so every file is where the cells below look for it. On your own computer, inside a copy of the labs, it does nothing.
import os, subprocess, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/tiny_repo"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/gkiri/tiny_repo", "/content/tiny_repo"], check=True)
    os.chdir("/content/tiny_repo/tiny-transformer/notebooks")
print("ready:", os.path.basename(os.getcwd()))

ready: notebooks


In [2]:
import base64, hashlib, json, math, os, urllib.request
import numpy as np
import torch

torch.manual_seed(0)
torch.set_grad_enabled(False)      # nothing is trained here, so no gradients are needed
print(torch.__version__)

2.10.0+cu128


### Setup: the text, the model and its trained numbers

The text gives the 65 characters their ids. The next cell is the model's whole code, unchanged; the one after it loads the
trained numbers into it.

In [3]:
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
SHA256 = "86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed"
here = os.path.abspath('')
candidates = [os.path.join(here, '..', 'data', 'tinyshakespeare.txt'),                                   # in the labs
              os.path.join(here, '..', '..', '..', '..', 'training', 'prompter', 'data', 'tinyshakespeare.txt')]   # in the course
found = [p for p in candidates if os.path.exists(p)]
raw = open(found[0], 'rb').read() if found else urllib.request.urlopen(URL, timeout=60).read()
assert hashlib.sha256(raw).hexdigest() == SHA256, "not the pinned Tiny Shakespeare"
text = raw.decode('utf-8')
print(f"{len(text):,} characters")

1,115,394 characters


In [4]:
import math
from dataclasses import asdict, dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass(frozen=True)
class Config:
    vocab_size: int = 65    # every distinct character in Tiny Shakespeare
    block_size: int = 32    # how many characters the model can read at once (its context)
    d_model: int = 28       # how many numbers describe one character in one place (a row)
    n_head: int = 2         # attention heads, each looking with 28 / 2 = 14 numbers
    d_ff: int = 56          # the feed-forward network's hidden width (2 × d_model)
    n_layer: int = 1        # transformer blocks

    def to_dict(self):
        return asdict(self)


class Tokenizer:
    """One id per character, in sorted order: '\\n' is 0, ' ' is 1, … 'z' is 64."""

    def __init__(self, text: str):
        self.chars = sorted(set(text))
        self.stoi = {c: i for i, c in enumerate(self.chars)}

    def encode(self, s: str) -> list[int]:
        return [self.stoi[c] for c in s]

    def decode(self, ids) -> str:
        return "".join(self.chars[int(i)] for i in ids)


class CausalSelfAttention(nn.Module):
    """Each row asks a question (q), every earlier row offers a label (k) and a message (v). A row reads the messages of the
    rows whose labels match its question best, never a row to its right."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.n_head, self.head_dim = cfg.n_head, cfg.d_model // cfg.n_head
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model)     # q, k and v for both heads in one matrix
        self.proj = nn.Linear(cfg.d_model, cfg.d_model)        # mix the heads' results back into one row
        mask = torch.tril(torch.ones(cfg.block_size, cfg.block_size))
        self.register_buffer("mask", mask.view(1, 1, cfg.block_size, cfg.block_size), persistent=False)

    def forward(self, x, keep=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.n_head, self.head_dim).transpose(1, 2) for t in (q, k, v))
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)          # how well each question matches each label
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))   # no peeking at the future
        att = F.softmax(att, dim=-1)                                          # shares that add up to 1
        if keep is not None:
            keep["attn"] = att.detach()
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)            # each row: a weighted mix of messages
        return self.proj(y)


class Block(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ff = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_ff), nn.GELU(), nn.Linear(cfg.d_ff, cfg.d_model))

    def forward(self, x, keep=None):
        x = x + self.attn(self.ln1(x), keep)    # look back, and add what was found to the row
        x = x + self.ff(self.ln2(x))            # think about each row on its own, and add that too
        return x


class Prompter(nn.Module):
    def __init__(self, cfg: Config = Config()):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)    # one learned row per character
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.d_model)    # one learned row per place 0 … 31
        self.blocks = nn.Sequential(*[Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head_bias = nn.Parameter(torch.zeros(cfg.vocab_size))  # the readout reuses tok_emb (tied); only its bias is new
        self.apply(self._init)

    @staticmethod
    def _init(m):
        # small random numbers (σ = 0.02), so the untrained model's guesses are nearly even: its first loss is about
        # ln 65 = 4.17, the loss of a uniform guess over 65 characters
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, keep=None):
        T = idx.size(1)
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))   # what + where
        for b in self.blocks:
            x = b(x, keep)
        x = self.ln_f(x)
        logits = x @ self.tok_emb.weight.T + self.head_bias                         # a score for every character
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, n, temperature=1.0, generator=None):
        for _ in range(n):
            logits, _ = self(idx[:, -self.cfg.block_size:])
            probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
            nxt = torch.multinomial(probs, 1, generator=generator)
            idx = torch.cat([idx, nxt], dim=1)
        return idx


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

In [5]:
paths = [os.path.join(here, '..', 'model', 'prompter.json'),                                                     # in the labs
         os.path.join(here, '..', '..', '..', '..', 'src', 'specimen', 'tiny-transformer', 'data', 'prompter.json')]   # in the course
found = [p for p in paths if os.path.exists(p)]
assert found, "the model's trained numbers are not here: run the Setup cell at the top"
exp = json.load(open(found[0]))
state = {t['name']: torch.tensor(np.frombuffer(base64.b64decode(t['data']), dtype='<f4').reshape(t['shape']).copy()) for t in exp['weights']['tensors']}
tok = Tokenizer(text)
model = Prompter(Config(vocab_size=len(tok.chars)))
model.load_state_dict(state)
model.eval()
block = model.blocks[0]                            # the model's one block
print('trained for', f"{exp['checkpoint_step']:,}", 'steps:', f"{count_params(model):,}", 'learned numbers')
assert exp['checkpoint_step'] == 98000 and count_params(model) == 9417


def same(computed, recorded, tol=2e-4):
    """True when every computed number is within tol of the course's recorded one (the record keeps 4 decimals)."""
    a = torch.as_tensor(computed, dtype=torch.float32).flatten()
    b = torch.tensor(recorded, dtype=torch.float32).flatten()
    return a.shape == b.shape and bool((a - b).abs().max() <= tol)

trained for 98,000 steps: 9,417 learned numbers


The episode's teaching code, `code/inside_readout.py`: the final LayerNorm, the logits and softmax, written out on the model's
own parts, and the whole forward pass one named stage at a time. Every comment gives a shape.

In [6]:
import math

import torch


def final_norm(model, x):
    """The final LayerNorm: each row to average 0 and spread 1, then a learned scale and shift. [T, 28] in, [T, 28] out."""
    ln = model.ln_f                                          # weight [28] (the scale) and bias [28] (the shift): 56 learned
    mean = x.mean(dim=-1, keepdim=True)                      # [T, 1]: each row's average
    var = ((x - mean) ** 2).mean(dim=-1, keepdim=True)       # [T, 1]: each row's spread, squared
    return (x - mean) / torch.sqrt(var + ln.eps) * ln.weight + ln.bias   # [T, 28]


def logits(model, rows):
    """A score for every character at every position: each row dotted with all 65 token rows, plus 65 biases."""
    table = model.tok_emb.weight                             # [65, 28]: the token table, the same one the pass starts with
    return rows @ table.T + model.head_bias                  # [T, 28] @ [28, 65] + [65] → [T, 65]


def softmax(z):
    """Scores into probabilities along the last axis: e to each score, divided by their total. Same shape in and out."""
    e = torch.exp(z - z.max(dim=-1, keepdim=True).values)   # subtracting the largest changes nothing, but keeps e^z small
    return e / e.sum(dim=-1, keepdim=True)                   # each row now adds up to 1


@torch.no_grad()
def forward_by_hand(model, ids):
    """The whole forward pass, one named stage at a time: ids [T] in, every stage's tensor out, in the order they run."""
    ids = torch.as_tensor(ids)                                # [T]: the characters as ids
    block = model.blocks[0]                                   # the model's one block
    T, C, H = len(ids), model.cfg.d_model, model.cfg.n_head  # T positions, 28 numbers per row, 2 heads
    Dh = C // H                                               # 14 numbers per head
    s = {"ids": ids}
    s["tok"] = model.tok_emb.weight[ids]                      # [T, 28]: each id picks its row of the token table
    s["pos"] = model.pos_emb.weight[:T]                       # [T, 28]: one row per place 0 … T−1
    s["x0"] = s["tok"] + s["pos"]                             # [T, 28]: the stream starts
    s["ln1"] = block.ln1(s["x0"])                             # [T, 28]: LayerNorm before attention
    q, k, v = block.attn.qkv(s["ln1"]).split(C, dim=1)        # each [T, 28]: queries, keys and values of both heads
    s["q"], s["k"], s["v"] = (t.view(T, H, Dh).transpose(0, 1) for t in (q, k, v))   # each [2, T, 14]: one per head
    s["scores"] = s["q"] @ s["k"].transpose(1, 2) / math.sqrt(Dh)                    # [2, T, T]: each query against each key
    future = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)              # [T, T]: True where a key is in the future
    s["shares"] = softmax(s["scores"].masked_fill(future, float("-inf")))            # [2, T, T]: each row adds up to 1
    s["mixed"] = s["shares"] @ s["v"]                         # [2, T, 14]: each row a weighted mix of values
    s["joined"] = s["mixed"].transpose(0, 1).reshape(T, C)    # [T, 28]: the two heads side by side
    s["proj"] = block.attn.proj(s["joined"])                  # [T, 28]: attention's update
    s["x1"] = s["x0"] + s["proj"]                             # [T, 28]: the first residual add
    s["ln2"] = block.ln2(s["x1"])                             # [T, 28]: LayerNorm before the feed-forward network
    s["hidden"] = block.ff[0](s["ln2"])                       # [T, 56]: each row widened
    s["gelu"] = block.ff[1](s["hidden"])                      # [T, 56]: bent
    s["ff"] = block.ff[2](s["gelu"])                          # [T, 28]: narrowed back: the feed-forward update
    s["x2"] = s["x1"] + s["ff"]                               # [T, 28]: the second residual add
    s["lnf"] = final_norm(model, s["x2"])                     # [T, 28]: the final LayerNorm
    s["logits"] = logits(model, s["lnf"])                     # [T, 65]: a score for every character at every position
    s["probs"] = softmax(s["logits"])                         # [T, 65]: probabilities; the last row predicts what comes next
    return s

### Where M4 left off

The line becomes 32 ids and rows `x0` (M2); the block, attention and then the feed-forward network, each added onto the stream
(M3, M4), gives `x2`, still [32×28].

In [7]:
LINE = "You taught me language; and my p"
ids = torch.tensor(tok.encode(LINE))                                  # [32]
x0 = model.tok_emb(ids) + model.pos_emb(torch.arange(len(ids)))       # [32, 28]: what + where (M2)
x2 = block(x0[None])[0]                                               # [32, 28]: the block's output (M3, M4)
print('ids', tuple(ids.shape), ' x0', tuple(x0.shape), ' x2', tuple(x2.shape))
assert same(x2[31, -4:], [0.7697, 1.4879, 3.521, 4.8752])                                      # row 31's last 4 numbers, as recorded

ids (32,)  x0 (32, 28)  x2 (32, 28)


## 1 · Scores for every character: the logits

First a final **LayerNorm** prepares the rows: 28 scales and 28 shifts, 56 learned numbers. Then each row is
compared with all 65 rows of the **token table**, the same [65×28] table that turned ids into rows at the start, by a dot
product (multiply number by number, and add), plus one learned bias per character (65). Reusing the table is
called **weight tying**: no second table to learn, only the 65 biases. The 65 scores of each row are its
**logits**, [32×65]. For row 31: r 10.67, a 10.19, l 9.73, o 9.50, e 9.49.

In [8]:
n = final_norm(model, x2)                          # [32, 28] → [32, 28]
print('final LayerNorm:', sum(p.numel() for p in model.ln_f.parameters()), 'learned numbers; rows', tuple(n.shape))
print('the token table:', tuple(model.tok_emb.weight.shape), ' the readout\'s own learned numbers: its bias', tuple(model.head_bias.shape))
z = logits(model, n)                               # [32, 28] @ [28, 65] + [65] → [32, 65]
print('logits', tuple(z.shape))
v, i = z[31].topk(5)
print("row 31's largest logits:", '   '.join(f"{tok.chars[k]!r} {s:.2f}" for s, k in zip(v.tolist(), i.tolist())))

assert torch.allclose(n, model.ln_f(x2), atol=1e-5)                                   # the model's own LayerNorm
assert sum(p.numel() for p in model.ln_f.parameters()) == 56 and model.head_bias.numel() == 65
assert [tuple(p.shape) for p in model.parameters()].count((65, 28)) == 1       # one [65×28] table, used twice: tied
assert tuple(z.shape) == (32, 65) and same(n[31, -4:], [0.3968, 0.5396, 3.2959, 6.18])
assert same(z.max(dim=1).values, [5.5792, 10.4717, 10.6764, 3.7299, 12.1115, 8.748, 10.5407, 11.6749, 12.0749, 8.8263, 4.0896, 11.1028, 9.3444, 4.379, 11.0122, 6.1578, 9.6172, 10.2052, 13.9962, 9.3117, 10.6931, 10.1164, 16.6986, 4.1682, 8.8581, 10.7789, 11.9515, 4.1264, 10.9435, 11.9236, 3.9339, 10.6675])                                            # every row's largest logit, as recorded
assert [tok.chars[k] for k in i.tolist()] == ["r", "a", "l", "o", "e"] and same(v, [10.6675, 10.1854, 9.7269, 9.5019, 9.486])

final LayerNorm: 56 learned numbers; rows (32, 28)
the token table: (65, 28)  the readout's own learned numbers: its bias (65,)
logits (32, 65)
row 31's largest logits: 'r' 10.67   'a' 10.19   'l' 9.73   'o' 9.50   'e' 9.49


### ✋ Your turn

Compute r's logit for row 31 yourself: the dot product of row 31 of `n` with r's row of the token table (multiply the 28 pairs, add them up), plus r's bias. Put it in `answer` as a number.

In [9]:
r = tok.stoi['r']
E = model.tok_emb.weight                           # [65, 28]: the token table
answer = None   # TODO: (n[31] * E[r]).sum().item() + model.head_bias[r].item()

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer - z[31, r].item()) < 1e-4, 'multiply n[31] and E[r] number by number, add the 28 products, then add model.head_bias[r]'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [10]:
#@title Show a solution { display-mode: "form" }
r = tok.stoi['r']
E = model.tok_emb.weight
answer = (n[31] * E[r]).sum().item() + model.head_bias[r].item()
print("r's logit for row 31:", round(answer, 4))

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer - z[31, r].item()) < 1e-4, 'multiply n[31] and E[r] number by number, add the 28 products, then add model.head_bias[r]'
    print("✓ right:", answer)

r's logit for row 31: 10.6675
✓ right: 10.667499452829361


## 2 · Softmax: from logits to probabilities

Logits can be any size, even negative, and they do not add up to anything in particular. **Softmax** turns a row of them into
probabilities: e to the power of each logit, divided by the total of all 65. The order stays the same, every number lands
between 0 and 1, and the row adds up to 1. For row 31: r 0.340, a 0.210, l 0.133, o 0.106, e 0.104.

In [11]:
p31 = softmax(z[31])                               # [65] → [65]
print("row 31's logits run from", round(z[31].min().item(), 2), 'to', round(z[31].max().item(), 2))
print('its probabilities run from', f"{p31.min().item():.1e}", 'to', round(p31.max().item(), 3), '; they add up to', round(p31.sum().item(), 6))
print(' char   logit   probability')
for k in z[31].topk(5).indices.tolist():
    print(f"  {tok.chars[k]!r}   {z[31, k].item():6.2f}    {p31[k].item():.3f}")

assert torch.allclose(p31, torch.softmax(z[31], dim=-1), atol=1e-7)                 # torch's own softmax
assert abs(p31.sum().item() - 1) < 1e-6 and bool(((p31 > 0) & (p31 < 1)).all())
assert torch.equal(p31.topk(10).indices, z[31].topk(10).indices)                     # the order stays the same
v, i = p31.topk(5)
assert [tok.chars[k] for k in i.tolist()] == ["r", "a", "l", "o", "e"] and same(v, [0.3403, 0.2101, 0.1328, 0.1061, 0.1044])

row 31's logits run from -17.96 to 10.67
its probabilities run from 1.3e-13 to 0.34 ; they add up to 1.0
 char   logit   probability
  'r'    10.67    0.340
  'a'    10.19    0.210
  'l'     9.73    0.133
  'o'     9.50    0.106
  'e'     9.49    0.104


### ✋ Your turn

How many times more probable is r than a? Softmax turns a gap between two logits into a ratio between their probabilities: e to the power of the gap. Work it out from the two logits alone and put it in `answer`; then compare it with `p31[r] / p31[a]`.

In [12]:
r, a = tok.stoi['r'], tok.stoi['a']
gap = (z[31, r] - z[31, a]).item()
answer = None   # TODO: math.exp(gap)

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer - (p31[r] / p31[a]).item()) < 1e-3, 'math.exp(gap): e to the power of the gap between the two logits'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [13]:
#@title Show a solution { display-mode: "form" }
r, a = tok.stoi['r'], tok.stoi['a']
gap = (z[31, r] - z[31, a]).item()
answer = math.exp(gap)
print('gap', round(gap, 2), '→ ratio', round(answer, 2), '; the probabilities:', round((p31[r] / p31[a]).item(), 2))

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer - (p31[r] / p31[a]).item()) < 1e-3, 'math.exp(gap): e to the power of the gap between the two logits'
    print("✓ right:", answer)

gap 0.48 → ratio 1.62 ; the probabilities: 1.62
✓ right: 1.6194277242524755


## 3 · Every position predicts

The readout ran on all 32 rows, not just the last: the probabilities are [32×65], and every row adds up to 1. Row t predicts
the character at t + 1, so the true next characters are the line shifted by one, plus the character that really follows the
line in the play: r. Training scores all 32 rows at once, 32 questions from one pass; writing uses only the last row.
After “; an”, the true next character, d, got 0.914; after “my ”, the true next, p, got only
0.026: many words could start there.

In [14]:
P = softmax(z)                                     # [32, 65]: one row of probabilities per position
nxt = text[text.find(LINE) + len(LINE)]            # the character that really follows the line in the play
truth = LINE[1:] + nxt                             # [32]: row t's true next character
y = torch.tensor(tok.encode(truth))                # [32]: as ids
p_true = P[torch.arange(32), y]                    # [32]: each row's probability of its true next character
show = lambda ch: {'\n': '↵', ' ': '␣'}.get(ch, ch)
print('probabilities', tuple(P.shape), '; every row adds up to 1:', bool(torch.allclose(P.sum(dim=1), torch.ones(32), atol=1e-6)))
print(' t  reads  true next  p(true next)  top guess')
for t in range(32):
    print(f"{t:2d}    {show(LINE[t])}        {show(truth[t])}         {p_true[t].item():.3f}         {show(tok.chars[int(P[t].argmax())])}")

RECORDED_P_TRUE = [0.1733, 0.7093, 0.2455, 0.0983, 0.017, 0.0043, 0.2172, 0.9314, 0.75, 0.5331, 0.0657, 0.2263, 0.4414, 0.0316, 0.1941, 0.1386, 0.0315, 0.0822, 0.1247, 0.0167, 0.5327, 0.0362, 0.2521, 0.1886, 0.6263, 0.9139, 0.9479, 0.0792, 0.2682, 0.9127, 0.0263, 0.3403]   # the course's record: each row's probability of its true next character
assert nxt == "r" and tuple(P.shape) == (32, 65) and torch.allclose(P.sum(dim=1), torch.ones(32), atol=1e-6)
assert same(p_true, RECORDED_P_TRUE)
assert truth[25] == "d" and f"{p_true[25]:.3f}" == "0.914" and truth[30] == "p" and f"{p_true[30]:.3f}" == "0.026"

probabilities (32, 65) ; every row adds up to 1: True
 t  reads  true next  p(true next)  top guess
 0    Y        o         0.173         o
 1    o        u         0.709         u
 2    u        ␣         0.246         r
 3    ␣        t         0.098         s
 4    t        a         0.017         h
 5    a        u         0.004         k
 6    u        g         0.217         s
 7    g        h         0.931         h
 8    h        t         0.750         t
 9    t        ␣         0.533         ␣
10    ␣        m         0.066         t
11    m        e         0.226         y
12    e        ␣         0.441         ␣
13    ␣        l         0.032         t
14    l        a         0.194         o
15    a        n         0.139         s
16    n        g         0.032         d
17    g        u         0.082         e
18    u        a         0.125         e
19    a        g         0.017         r
20    g        e         0.533         e
21    e        ;         0.036         

Training scores every row with M1's score, −ln p of the true next character, and averages them: that average is the **loss**
`model.py` returns when it is given the targets. And the **causal mask** makes all 32 questions fair: change the line's last
character and only the last row's prediction changes, because no row can see the characters after it.

In [15]:
loss = -p_true.log().mean()                        # −ln p, averaged over the 32 questions
_, model_loss = model(ids[None], y[None])          # the model's own loss, given the 32 targets
print('the average of −ln p over the 32 rows:', round(loss.item(), 4), '  model.py:', round(model_loss.item(), 4))

other = LINE[:-1] + 'b'                            # the same line with its last character changed
P_other = softmax(model(torch.tensor([tok.encode(other)]))[0][0])        # [32, 65]
P_line = softmax(model(ids[None])[0][0])                                  # [32, 65]
rows_changed = torch.nonzero((P_other - P_line).abs().amax(dim=1) > 0).flatten().tolist()
print('rows whose prediction changed:', rows_changed)

assert torch.allclose(loss, model_loss, atol=1e-5) and abs(loss.item() - 1.8235) < 2e-3
assert rows_changed == [31]

the average of −ln p over the 32 rows: 1.8234   model.py: 1.8234


rows whose prediction changed: [31]


### ✋ Your turn

Which position gave its true next character the highest probability? Put the position t (0 to 31) in `answer`, then look up which character it read and which came next.

In [16]:
answer = None   # TODO: int(p_true.argmax())

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == int(p_true.argmax()), 'int(p_true.argmax()): the position of the largest number in p_true'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [17]:
#@title Show a solution { display-mode: "form" }
answer = int(p_true.argmax())
print('row', answer, 'reads', repr(LINE[answer]), 'and gave the true next,', repr(truth[answer]), ',', round(p_true[answer].item(), 3))

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == int(p_true.argmax()), 'int(p_true.argmax()): the position of the largest number in p_true'
    print("✓ right:", answer)

row 26 reads 'd' and gave the true next, ' ' , 0.948
✓ right: 26


## 4 · The whole pass

Every stage is open now. `forward_by_hand` runs them all in order, from the 32 ids to the probabilities, with the model's own
learned numbers, and returns every stage: the token rows and position rows; attention (each head's queries, keys and values,
scores, shares and mix, then the heads joined and projected) and the first add; the feed-forward network and the second add;
the final LayerNorm, the logits and the probabilities. Each stage is printed with its shape and checked against the course's
record (the last numbers of row 31; head 0's for the per-head stages). The last stage must equal the model's own forward
pass.

In [18]:
S = forward_by_hand(model, ids)
RECORD = {"ids": [[32], [51, 63, 1, 54]], "tok": [[32, 28], [-0.0007, 1.1266, 0.1156, -0.0086]], "pos": [[32, 28], [0.256, -0.1686, -0.6599, 0.1745]], "x0": [[32, 28], [0.2553, 0.9579, -0.5443, 0.166]], "ln1": [[32, 28], [0.0332, 1.5467, -2.0076, 0.2312]], "q": [[2, 32, 14], [-1.2342, -4.2584, 3.1272, -0.5446]], "k": [[2, 32, 14], [-3.9774, -5.4123, 5.0737, -0.585]], "v": [[2, 32, 14], [1.1884, -1.4838, -0.3599, -0.5915]], "scores": [[2, 32, 32], [21.813, 23.1813, 26.0642, 23.2109]], "shares": [[2, 32, 32], [0.0125, 0.0492, 0.8782, 0.0506]], "mixed": [[2, 32, 14], [0.1864, -0.1336, 0.0386, -0.3272]], "joined": [[32, 28], [1.0638, -0.1869, 0.1583, -0.1342]], "proj": [[32, 28], [-0.202, -0.2227, 0.0633, 0.5605]], "x1": [[32, 28], [0.0533, 0.7352, -0.4811, 0.7264]], "ln2": [[32, 28], [-0.1136, 1.8061, -1.2529, 0.9633]], "hidden": [[32, 56], [-7.3999, -2.2145, -6.009, 3.576]], "gelu": [[32, 56], [-0.0, -0.0297, -0.0, 3.5754]], "ff": [[32, 28], [0.7164, 0.7527, 4.002, 4.1488]], "x2": [[32, 28], [0.7697, 1.4879, 3.521, 4.8752]], "lnf": [[32, 28], [0.3968, 0.5396, 3.2959, 6.18]], "logits": [[32, 65], [3.8505, 2.7287, 5.8219, -5.201]], "probs": [[32, 65], [0.0004, 0.0001, 0.0027, 0.0]]}   # the course's record of every stage: its shape, and the last 4 numbers of row 31
print(' stage    shape          row 31 ends with')
for name, t in S.items():
    row = t if t.dim() == 1 else (t[31] if t.dim() == 2 else t[0, 31])   # ids; a [T, …] stage; a per-head stage: head 0
    print(f"{name:8s} {str(tuple(t.shape)):14s} {[round(x, 3) for x in row[-4:].tolist()]}")
    shape, last4 = RECORD[name]
    assert tuple(t.shape) == tuple(shape) and same(row[-4:], last4), name
assert list(S) == list(RECORD)                     # every stage, in the recorded order

 stage    shape          row 31 ends with
ids      (32,)          [51, 63, 1, 54]
tok      (32, 28)       [-0.001, 1.127, 0.116, -0.009]
pos      (32, 28)       [0.256, -0.169, -0.66, 0.175]
x0       (32, 28)       [0.255, 0.958, -0.544, 0.166]
ln1      (32, 28)       [0.033, 1.547, -2.008, 0.231]
q        (2, 32, 14)    [-1.234, -4.258, 3.127, -0.545]
k        (2, 32, 14)    [-3.977, -5.412, 5.074, -0.585]
v        (2, 32, 14)    [1.188, -1.484, -0.36, -0.592]
scores   (2, 32, 32)    [21.813, 23.181, 26.064, 23.211]
shares   (2, 32, 32)    [0.013, 0.049, 0.878, 0.051]
mixed    (2, 32, 14)    [0.186, -0.134, 0.039, -0.327]
joined   (32, 28)       [1.064, -0.187, 0.158, -0.134]
proj     (32, 28)       [-0.202, -0.223, 0.063, 0.56]
x1       (32, 28)       [0.053, 0.735, -0.481, 0.726]
ln2      (32, 28)       [-0.114, 1.806, -1.253, 0.963]
hidden   (32, 56)       [-7.4, -2.215, -6.009, 3.576]
gelu     (32, 56)       [-0.0, -0.03, -0.0, 3.575]
ff       (32, 28)       [0.716, 0.753, 4.002, 

In [19]:
own, _ = model(ids[None])                          # the model's own forward pass: [1, 32, 65]
gap = (S['logits'] - own[0]).abs().max().item()
print('largest gap between the logits by hand and the model\'s own:', f"{gap:.1e}")
chain = ['[' + '×'.join(str(n) for n in S[k].shape) + ']' for k in ('ids', 'x0', 'q', 'scores', 'x1', 'hidden', 'x2', 'logits')]
last = S['probs'][-1]                              # [65]: the last row, what comes after the line
print(' → '.join(chain), '→ [65] →', repr(tok.chars[int(last.argmax())]), round(last.max().item(), 3))
assert torch.allclose(S['logits'], own[0], atol=1e-5)
assert tok.chars[int(last.argmax())] == "r" and same(last.max(), 0.3403)

largest gap between the logits by hand and the model's own: 6.7e-06
[32] → [32×28] → [2×32×14] → [2×32×32] → [32×28] → [32×56] → [32×28] → [32×65] → [65] → 'r' 0.34


### ✋ Your turn

Every 32 in those shapes is the line's length: the model reads up to 32 characters, and a shorter line runs the same way. Run the whole pass on the first 10 characters, `"You taught"`, and put the shape of its logits in `answer`, as a tuple.

In [20]:
short = forward_by_hand(model, tok.encode("You taught"))
answer = None   # TODO: tuple(short['logits'].shape)

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == (10, 65), "tuple(short['logits'].shape): one row per character, one score per character of the vocabulary"
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [21]:
#@title Show a solution { display-mode: "form" }
short = forward_by_hand(model, tok.encode("You taught"))
for name in ('ids', 'x0', 'q', 'scores', 'hidden', 'logits'):
    print(f"{name:7s}", tuple(short[name].shape))
answer = tuple(short['logits'].shape)

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == (10, 65), "tuple(short['logits'].shape): one row per character, one score per character of the vocabulary"
    print("✓ right:", answer)

ids     (10,)
x0      (10, 28)
q       (2, 10, 14)
scores  (2, 10, 10)
hidden  (10, 56)
logits  (10, 65)
✓ right: (10, 65)
